
# SIT720 - 10.2 Credit Task
## Neural Networks for Image Classification using Fashion-MNIST

**Student:** Zenith Anthony

This notebook addresses all five assessment sections:

1. Dataset understanding and baseline MLP  
2. Effect of network depth  
3. Effect of network width  
4. Depth vs width comparison  
5. Hyperparameter tuning, error analysis, and final recommendation  

> **Important:** Run the notebook from top to bottom in Google Colab. It downloads Fashion-MNIST if it is not already cached, executes all experiments, displays outputs/figures, and generates a final PDF report automatically from the actual results.


In [ ]:

# Core imports
import os, sys, time, json, gzip, urllib.request, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)



## 1. Load and understand Fashion-MNIST

Fashion-MNIST contains 60,000 training images and 10,000 test images.  
Each image is a 28 x 28 grayscale image and belongs to one of 10 clothing classes.

The loader below first tries `tensorflow.keras.datasets.fashion_mnist`. If TensorFlow is not available, it downloads the official four gzip files and parses them directly with NumPy.


In [ ]:

def load_fashion_mnist():
    # Preferred option: TensorFlow/Keras
    try:
        from tensorflow.keras.datasets import fashion_mnist
        (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
        return x_train, y_train, x_test, y_test
    except Exception as e:
        print("Keras loader unavailable; using direct official download.")

    base = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
    files = {
        "train_labels": "train-labels-idx1-ubyte.gz",
        "train_images": "train-images-idx3-ubyte.gz",
        "test_labels": "t10k-labels-idx1-ubyte.gz",
        "test_images": "t10k-images-idx3-ubyte.gz",
    }
    data_dir = Path("fashion_mnist_data")
    data_dir.mkdir(exist_ok=True)

    for fname in files.values():
        fp = data_dir / fname
        if not fp.exists():
            print("Downloading", fname)
            urllib.request.urlretrieve(base + fname, fp)

    def read_labels(path):
        with gzip.open(path, "rb") as f:
            return np.frombuffer(f.read(), dtype=np.uint8, offset=8)

    def read_images(path):
        with gzip.open(path, "rb") as f:
            data = np.frombuffer(f.read(), dtype=np.uint8, offset=16)
            return data.reshape(-1, 28, 28)

    y_train = read_labels(data_dir / files["train_labels"])
    x_train = read_images(data_dir / files["train_images"])
    y_test = read_labels(data_dir / files["test_labels"])
    x_test = read_images(data_dir / files["test_images"])
    return x_train, y_train, x_test, y_test

x_train_img, y_train, x_test_img, y_test = load_fashion_mnist()

print("Training images:", x_train_img.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test_img.shape)
print("Testing labels:", y_test.shape)
print("Image dimensions:", x_train_img.shape[1:])
print("Classes:", list(enumerate(CLASS_NAMES)))


In [ ]:

# Representative examples - one image per class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls, ax in enumerate(axes.ravel()):
    idx = np.where(y_train == cls)[0][0]
    ax.imshow(x_train_img[idx], cmap="gray")
    ax.set_title(f"{cls}: {CLASS_NAMES[cls]}")
    ax.axis("off")
plt.suptitle("Representative Fashion-MNIST Images")
plt.tight_layout()
plt.show()

# Flatten and scale pixels to [0, 1]
X_train = x_train_img.reshape(len(x_train_img), -1).astype("float32") / 255.0
X_test = x_test_img.reshape(len(x_test_img), -1).astype("float32") / 255.0

print("Flattened training shape:", X_train.shape)
print("Flattened test shape:", X_test.shape)



### Classification challenges

Fashion-MNIST is more difficult than handwritten-digit MNIST because several clothing categories have similar silhouettes.  
Typical confusions include **shirt vs T-shirt/top**, **shirt vs pullover**, and **coat vs pullover**. An MLP also discards the explicit 2-D spatial structure when each 28 x 28 image is flattened into 784 input features.


In [ ]:

# Helper for consistent experiments
def fit_mlp(hidden_layers, activation="relu", solver="adam", alpha=1e-4,
            batch_size=256, learning_rate_init=1e-3, max_iter=30,
            early_stopping=True, Xtr=None, ytr=None, Xte=None, yte=None):
    if Xtr is None: Xtr, ytr, Xte, yte = X_train, y_train, X_test, y_test

    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        solver=solver,
        alpha=alpha,
        batch_size=batch_size,
        learning_rate_init=learning_rate_init,
        max_iter=max_iter,
        early_stopping=early_stopping,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=RANDOM_STATE
    )
    start = time.perf_counter()
    model.fit(Xtr, ytr)
    elapsed = time.perf_counter() - start

    train_acc = accuracy_score(ytr, model.predict(Xtr))
    test_acc = accuracy_score(yte, model.predict(Xte))
    return model, train_acc, test_acc, elapsed

results = {}



## 1A. Baseline MLP

Architecture:

- **Input layer:** 784 normalized pixel values (28 x 28 flattened image)
- **Hidden layer:** 100 neurons with ReLU activation
- **Output:** 10-class classification handled internally by `MLPClassifier`

The hidden layer learns non-linear combinations of pixel inputs. The output layer produces scores/probabilities used to predict one of the 10 clothing classes.


In [ ]:

baseline_model, baseline_train_acc, baseline_test_acc, baseline_time = fit_mlp((100,))
results["baseline"] = {
    "hidden_layers": [100],
    "train_accuracy": baseline_train_acc,
    "test_accuracy": baseline_test_acc,
    "training_time_s": baseline_time,
    "iterations": int(baseline_model.n_iter_)
}
pd.DataFrame([results["baseline"]])



**Baseline strengths:** simple, fast to define, non-linear, and substantially more expressive than a linear classifier.  
**Baseline limitations:** flattening removes spatial locality; the model has many dense connections; it can confuse visually similar classes; and MLPs are generally less suitable than convolutional neural networks for images.


## 2. Investigating the effect of network depth

In [ ]:

depths = [2, 4, 6, 8, 10]
depth_rows = []
depth_models = {}

for d in depths:
    layers = tuple([100] * d)
    print(f"Training depth={d}, layers={layers}")
    model, tr_acc, te_acc, elapsed = fit_mlp(layers)
    depth_models[d] = model
    depth_rows.append({
        "hidden_layers": d,
        "neurons_per_layer": 100,
        "train_accuracy": tr_acc,
        "test_accuracy": te_acc,
        "training_time_s": elapsed,
        "iterations": model.n_iter_
    })

depth_df = pd.DataFrame(depth_rows)
results["depth"] = depth_rows
display(depth_df)

plt.figure(figsize=(8, 5))
plt.plot(depth_df["hidden_layers"], depth_df["train_accuracy"], marker="o", label="Training accuracy")
plt.plot(depth_df["hidden_layers"], depth_df["test_accuracy"], marker="o", label="Test accuracy")
plt.xlabel("Number of hidden layers")
plt.ylabel("Accuracy")
plt.title("Effect of Network Depth")
plt.xticks(depths)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(depth_df["hidden_layers"], depth_df["training_time_s"], marker="o")
plt.xlabel("Number of hidden layers")
plt.ylabel("Training time (seconds)")
plt.title("Training Time vs Network Depth")
plt.xticks(depths)
plt.grid(alpha=0.3)
plt.show()



### Interpretation guide for depth

Use the generated table and graph to discuss whether test accuracy improves consistently as depth increases.  
A deeper dense network has greater representational capacity, but it is also harder to optimise and computationally more expensive. If training accuracy increases while test accuracy stops improving or decreases, that is evidence of reduced generalisation/overfitting.


## 3. Investigating the effect of network width

In [ ]:

widths = [20, 50, 100, 200]
width_rows = []
width_models = {}

for w in widths:
    layers = (w, w)  # fixed depth = 2 hidden layers
    print(f"Training width={w}, layers={layers}")
    model, tr_acc, te_acc, elapsed = fit_mlp(layers)
    width_models[w] = model
    width_rows.append({
        "hidden_layers": 2,
        "neurons_per_layer": w,
        "train_accuracy": tr_acc,
        "test_accuracy": te_acc,
        "training_time_s": elapsed,
        "iterations": model.n_iter_
    })

width_df = pd.DataFrame(width_rows)
results["width"] = width_rows
display(width_df)

plt.figure(figsize=(8, 5))
plt.plot(width_df["neurons_per_layer"], width_df["train_accuracy"], marker="o", label="Training accuracy")
plt.plot(width_df["neurons_per_layer"], width_df["test_accuracy"], marker="o", label="Test accuracy")
plt.xlabel("Neurons per hidden layer")
plt.ylabel("Accuracy")
plt.title("Effect of Network Width (2 Hidden Layers)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(width_df["neurons_per_layer"], width_df["training_time_s"], marker="o")
plt.xlabel("Neurons per hidden layer")
plt.ylabel("Training time (seconds)")
plt.title("Training Time vs Network Width")
plt.grid(alpha=0.3)
plt.show()



## 4. Comparing depth and width

The following cell automatically identifies the highest test accuracy observed in the controlled depth and width experiments.  
The discussion should consider **accuracy, training time, parameter growth, optimisation difficulty, and generalisation**, not accuracy alone.


In [ ]:

best_depth_row = depth_df.loc[depth_df["test_accuracy"].idxmax()]
best_width_row = width_df.loc[width_df["test_accuracy"].idxmax()]

comparison_df = pd.DataFrame([
    {
        "experiment": "Best depth experiment",
        "architecture": f"{int(best_depth_row.hidden_layers)} x 100",
        "test_accuracy": best_depth_row.test_accuracy,
        "training_time_s": best_depth_row.training_time_s,
    },
    {
        "experiment": "Best width experiment",
        "architecture": f"2 x {int(best_width_row.neurons_per_layer)}",
        "test_accuracy": best_width_row.test_accuracy,
        "training_time_s": best_width_row.training_time_s,
    }
])
display(comparison_df)

print("Best depth configuration:")
display(best_depth_row.to_frame().T)
print("\nBest width configuration:")
display(best_width_row.to_frame().T)



## 5. Hyperparameter tuning, error analysis, and recommendation

To avoid choosing a final model directly from the test set, this section creates a **training/validation split from the original training data**.  
The validation split is used for hyperparameter selection, and the untouched Fashion-MNIST test set is used only for the final evaluation.

Additional hyperparameters investigated:

- **Activation:** `relu` vs `tanh`
- **L2 regularisation (`alpha`):** `0.0001` vs `0.001`

A compact 2 x 100 architecture is used so that the tuning experiment remains computationally manageable.


In [ ]:

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=RANDOM_STATE
)

tuning_rows = []
for activation in ["relu", "tanh"]:
    for alpha in [1e-4, 1e-3]:
        model = MLPClassifier(
            hidden_layer_sizes=(100, 100),
            activation=activation,
            solver="adam",
            alpha=alpha,
            batch_size=256,
            learning_rate_init=1e-3,
            max_iter=30,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=5,
            random_state=RANDOM_STATE
        )
        start = time.perf_counter()
        model.fit(X_fit, y_fit)
        elapsed = time.perf_counter() - start
        fit_acc = accuracy_score(y_fit, model.predict(X_fit))
        val_acc = accuracy_score(y_val, model.predict(X_val))
        tuning_rows.append({
            "activation": activation,
            "alpha": alpha,
            "train_accuracy": fit_acc,
            "validation_accuracy": val_acc,
            "training_time_s": elapsed,
            "iterations": model.n_iter_
        })

tuning_df = pd.DataFrame(tuning_rows).sort_values("validation_accuracy", ascending=False)
results["tuning"] = tuning_rows
display(tuning_df)

best_tune = tuning_df.iloc[0]
print("Selected from validation evidence:")
display(best_tune.to_frame().T)


In [ ]:

# Retrain selected hyperparameters on the full training set, then test once
final_model = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation=best_tune["activation"],
    solver="adam",
    alpha=float(best_tune["alpha"]),
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=RANDOM_STATE
)

start = time.perf_counter()
final_model.fit(X_train, y_train)
final_time = time.perf_counter() - start

y_pred = final_model.predict(X_test)
final_test_acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Final test accuracy: {final_test_acc:.4f}")
print(f"Final training time: {final_time:.2f} s")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

fig, ax = plt.subplots(figsize=(9, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Final Model - Confusion Matrix")
plt.tight_layout()
plt.show()

results["final"] = {
    "architecture": [100, 100],
    "activation": str(best_tune["activation"]),
    "alpha": float(best_tune["alpha"]),
    "test_accuracy": final_test_acc,
    "training_time_s": final_time,
    "iterations": int(final_model.n_iter_)
}


In [ ]:

# Identify most frequently confused class pairs
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)

pairs = []
for true_cls in range(10):
    pred_cls = int(np.argmax(cm_off[true_cls]))
    pairs.append({
        "true_class": CLASS_NAMES[true_cls],
        "most_common_wrong_prediction": CLASS_NAMES[pred_cls],
        "count": int(cm_off[true_cls, pred_cls])
    })

pair_df = pd.DataFrame(pairs).sort_values("count", ascending=False)
display(pair_df.head(10))
results["misclassification_pairs"] = pair_df.to_dict("records")


In [ ]:

# Display correct and incorrect prediction examples
correct_idx = np.where(y_pred == y_test)[0]
wrong_idx = np.where(y_pred != y_test)[0]

rng = np.random.default_rng(RANDOM_STATE)
correct_sample = rng.choice(correct_idx, size=min(6, len(correct_idx)), replace=False)
wrong_sample = rng.choice(wrong_idx, size=min(6, len(wrong_idx)), replace=False)

fig, axes = plt.subplots(2, 6, figsize=(14, 6))
for j, idx in enumerate(correct_sample):
    axes[0, j].imshow(x_test_img[idx], cmap="gray")
    axes[0, j].set_title(f"Correct\n{CLASS_NAMES[y_test[idx]]}")
    axes[0, j].axis("off")

for j, idx in enumerate(wrong_sample):
    axes[1, j].imshow(x_test_img[idx], cmap="gray")
    axes[1, j].set_title(
        f"True: {CLASS_NAMES[y_test[idx]]}\nPred: {CLASS_NAMES[y_pred[idx]]}",
        fontsize=9
    )
    axes[1, j].axis("off")

plt.suptitle("Examples of Correct and Incorrect Predictions")
plt.tight_layout()
plt.show()



### Limitations of MLPs for image classification

1. **Loss of spatial structure:** flattening an image treats nearby and distant pixels similarly.
2. **Large number of dense parameters:** fully connected layers can become expensive as width/depth increases.
3. **No built-in translation/locality bias:** unlike CNNs, an MLP does not explicitly exploit local patterns such as edges and textures.
4. **Optimisation sensitivity:** deeper MLPs may be harder to train and can be sensitive to architecture and hyperparameters.
5. **Confusion between visually similar garments:** classes with similar outlines can overlap substantially in pixel space.

A reasonable recommendation is therefore the smallest architecture that achieves near-best validation/test performance rather than simply the deepest or widest model.


In [ ]:

# Automatically generate evidence-based summary statements
best_depth = depth_df.loc[depth_df["test_accuracy"].idxmax()]
best_width = width_df.loc[width_df["test_accuracy"].idxmax()]

print("=== AUTOMATIC SUMMARY ===")
print(f"Baseline test accuracy: {baseline_test_acc:.4f}")
print(f"Best depth: {int(best_depth.hidden_layers)} hidden layers x 100 neurons "
      f"(test accuracy={best_depth.test_accuracy:.4f}, time={best_depth.training_time_s:.2f}s)")
print(f"Best width: 2 hidden layers x {int(best_width.neurons_per_layer)} neurons "
      f"(test accuracy={best_width.test_accuracy:.4f}, time={best_width.training_time_s:.2f}s)")
print(f"Selected tuning: activation={best_tune.activation}, alpha={best_tune.alpha}, "
      f"validation accuracy={best_tune.validation_accuracy:.4f}")
print(f"Final test accuracy: {final_test_acc:.4f}")
print("\nMost common confusion:")
display(pair_df.head(5))



## GenAI acknowledgement

Generative AI was used for planning the notebook structure, suggesting code organisation, and improving wording/grammar. The code, outputs, model behaviour, and conclusions must be checked and understood by the student before submission. Experimental results in the final report are generated by executing this notebook rather than being invented.


In [ ]:

# Save machine-readable results
with open("SIT720_10_2C_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved: SIT720_10_2C_results.json")



## Automatic PDF report generation

The next cell creates a concise PDF report using the **actual results obtained during this run**.  
After execution, download the generated file together with this notebook.


In [ ]:

# Generate the final PDF report from the actual executed results.
# If ReportLab is unavailable in Colab, install it automatically.
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
except ImportError:
    !pip -q install reportlab
    from reportlab.lib.pagesizes import A4
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

from reportlab.lib.enums import TA_CENTER

report_name = "SIT720_10_2C_Final_Report_Zenith_Anthony.pdf"
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="CenterTitle", parent=styles["Title"], alignment=TA_CENTER))
styles.add(ParagraphStyle(name="Small", parent=styles["BodyText"], fontSize=9, leading=12))
styles.add(ParagraphStyle(name="Body2", parent=styles["BodyText"], fontSize=10, leading=14))

doc = SimpleDocTemplate(report_name, pagesize=A4, rightMargin=42, leftMargin=42, topMargin=42, bottomMargin=42)
story = []

story += [
    Paragraph("SIT720 - 10.2 Credit Task", styles["CenterTitle"]),
    Paragraph("Neural Networks for Image Classification using Fashion-MNIST", styles["Heading2"]),
    Paragraph("Student: Zenith Anthony", styles["BodyText"]),
    Spacer(1, 12),
    Paragraph("<b>1. Dataset and Baseline Neural Network</b>", styles["Heading2"]),
    Paragraph(
        "Fashion-MNIST contains 60,000 training images and 10,000 test images. "
        "Each example is a 28 x 28 grayscale image belonging to one of ten clothing classes. "
        "The images were flattened to 784 features and pixel values were scaled to [0,1]. "
        "The baseline MLP used one hidden layer of 100 ReLU neurons. The input layer represents "
        "the 784 pixels, the hidden layer learns non-linear feature combinations, and the output "
        "represents the ten-class prediction.",
        styles["Body2"]
    ),
    Paragraph(
        f"The baseline reached training accuracy {baseline_train_acc:.4f} and test accuracy "
        f"{baseline_test_acc:.4f} in {baseline_time:.2f} seconds. Its strengths are simplicity "
        "and non-linear modelling capacity. Its main limitation is that flattening removes image "
        "spatial structure, so visually similar classes can remain difficult to separate.",
        styles["Body2"]
    ),
    Spacer(1, 10),
    Paragraph("<b>2. Effect of Network Depth</b>", styles["Heading2"]),
]

depth_table = [["Hidden layers", "Neurons/layer", "Train acc.", "Test acc.", "Time (s)"]]
for _, r in depth_df.iterrows():
    depth_table.append([
        str(int(r.hidden_layers)), "100",
        f"{r.train_accuracy:.4f}", f"{r.test_accuracy:.4f}",
        f"{r.training_time_s:.2f}"
    ])
t = Table(depth_table, repeatRows=1, colWidths=[75, 85, 75, 75, 70])
t.setStyle(TableStyle([
    ("BACKGROUND",(0,0),(-1,0),colors.lightgrey),
    ("GRID",(0,0),(-1,-1),0.5,colors.grey),
    ("ALIGN",(0,0),(-1,-1),"CENTER"),
    ("FONTSIZE",(0,0),(-1,-1),8),
]))
story += [t, Spacer(1, 8)]

depth_best = depth_df.loc[depth_df.test_accuracy.idxmax()]
story.append(Paragraph(
    f"The strongest depth configuration in this experiment used {int(depth_best.hidden_layers)} "
    f"hidden layers with 100 neurons per layer and achieved test accuracy "
    f"{depth_best.test_accuracy:.4f}. The results show whether added depth produced consistent "
    "generalisation gains: increasing depth increases model capacity, but also increases "
    "optimisation difficulty and computation. Any widening gap between training and test accuracy "
    "indicates reduced generalisation.",
    styles["Body2"]
))

story += [Spacer(1, 10), Paragraph("<b>3. Effect of Network Width</b>", styles["Heading2"])]
width_table = [["Hidden layers", "Neurons/layer", "Train acc.", "Test acc.", "Time (s)"]]
for _, r in width_df.iterrows():
    width_table.append([
        "2", str(int(r.neurons_per_layer)),
        f"{r.train_accuracy:.4f}", f"{r.test_accuracy:.4f}",
        f"{r.training_time_s:.2f}"
    ])
t2 = Table(width_table, repeatRows=1, colWidths=[75, 85, 75, 75, 70])
t2.setStyle(TableStyle([
    ("BACKGROUND",(0,0),(-1,0),colors.lightgrey),
    ("GRID",(0,0),(-1,-1),0.5,colors.grey),
    ("ALIGN",(0,0),(-1,-1),"CENTER"),
    ("FONTSIZE",(0,0),(-1,-1),8),
]))
story += [t2, Spacer(1, 8)]

width_best = width_df.loc[width_df.test_accuracy.idxmax()]
story.append(Paragraph(
    f"The strongest width configuration used two hidden layers with "
    f"{int(width_best.neurons_per_layer)} neurons per layer and achieved test accuracy "
    f"{width_best.test_accuracy:.4f}. Wider networks can learn more features, but gains may "
    "saturate while training cost and overfitting risk continue to increase.",
    styles["Body2"]
))

story += [
    Spacer(1, 10),
    Paragraph("<b>4. Depth versus Width</b>", styles["Heading2"]),
    Paragraph(
        f"The best depth experiment achieved {depth_best.test_accuracy:.4f} test accuracy "
        f"in {depth_best.training_time_s:.2f} s, while the best width experiment achieved "
        f"{width_best.test_accuracy:.4f} in {width_best.training_time_s:.2f} s. "
        "The comparison therefore considers both predictive performance and computational cost. "
        "The practical design implication is to prefer the least complex architecture that obtains "
        "near-best generalisation rather than assuming that deeper or wider is always better.",
        styles["Body2"]
    ),
    Spacer(1, 10),
    Paragraph("<b>5. Hyperparameter Tuning, Error Analysis and Recommendation</b>", styles["Heading2"]),
    Paragraph(
        "Two additional hyperparameters were evaluated: activation function (ReLU and tanh) "
        "and L2 regularisation alpha (0.0001 and 0.001). Model selection was performed on a "
        "validation split derived only from the training data, so the test set was kept for final reporting.",
        styles["Body2"]
    )
]

tune_table = [["Activation", "Alpha", "Train acc.", "Validation acc.", "Time (s)"]]
for _, r in tuning_df.iterrows():
    tune_table.append([
        str(r.activation), f"{r.alpha:.4g}",
        f"{r.train_accuracy:.4f}", f"{r.validation_accuracy:.4f}",
        f"{r.training_time_s:.2f}"
    ])
t3 = Table(tune_table, repeatRows=1, colWidths=[70, 60, 75, 90, 70])
t3.setStyle(TableStyle([
    ("BACKGROUND",(0,0),(-1,0),colors.lightgrey),
    ("GRID",(0,0),(-1,-1),0.5,colors.grey),
    ("ALIGN",(0,0),(-1,-1),"CENTER"),
    ("FONTSIZE",(0,0),(-1,-1),8),
]))
story += [t3, Spacer(1, 8)]

story.append(Paragraph(
    f"The selected configuration used activation={best_tune.activation} and alpha={best_tune.alpha}, "
    f"with validation accuracy {best_tune.validation_accuracy:.4f}. After retraining on the full "
    f"training set, the final test accuracy was {final_test_acc:.4f}. The most frequent confusion "
    f"was {pair_df.iloc[0].true_class} predicted as {pair_df.iloc[0].most_common_wrong_prediction} "
    f"({int(pair_df.iloc[0]['count'])} cases). This is consistent with the visual similarity of "
    "some garment categories.",
    styles["Body2"]
))

story.append(Paragraph(
    "MLP limitations for image classification include loss of spatial structure after flattening, "
    "large dense parameter counts, sensitivity to depth/width/hyperparameters, and no built-in "
    "translation or locality bias. A convolutional neural network would normally be a stronger image-specific "
    "choice. Within the required MLP family, the recommended model is the validation-selected two-hidden-layer "
    f"(100, 100) network using {best_tune.activation} activation and alpha={best_tune.alpha}, because "
    "the recommendation is based on training-only validation evidence and is then evaluated once on the test set.",
    styles["Body2"]
))

story += [
    Spacer(1, 10),
    Paragraph("<b>GenAI acknowledgement</b>", styles["Heading2"]),
    Paragraph(
        "Generative AI was used for planning, code organisation, and language editing. "
        "The notebook was executed to obtain the numerical results, and the student is responsible "
        "for reviewing, understanding, and explaining all code, outputs, conclusions, and limitations.",
        styles["Body2"]
    )
]

doc.build(story)
print("Generated final report:", report_name)
